<a href="https://colab.research.google.com/github/gusainnikhil02-create/Oriented-Evaluation-Framework-of-Mobile-Learning-Apps/blob/main/Project_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install streamlit pandas plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 67.9 MB/s eta 0:00:00


In [2]:
import streamlit as st
import sqlite3
import pandas as pd
import plotly.express as px

# -------------------------
# DATABASE
# -------------------------

conn = sqlite3.connect("evaluation.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS apps(
id INTEGER PRIMARY KEY AUTOINCREMENT,
name TEXT,
category TEXT,
age TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS evaluation(
id INTEGER PRIMARY KEY AUTOINCREMENT,
app TEXT,
education INTEGER,
accessibility INTEGER,
usability INTEGER,
offline INTEGER,
language INTEGER,
community INTEGER,
total INTEGER
)
""")

conn.commit()

# -------------------------
# PAGE
# -------------------------

st.set_page_config(
page_title="Mobile Learning Evaluation",
layout="wide"
)

st.title("📚 Mobile Learning App Evaluation System")

menu = st.sidebar.selectbox(
"Menu",
[
"Dashboard",
"Add App",
"Evaluate App",
"View Results"
]
)

# -------------------------
# ADD APP
# -------------------------

if menu=="Add App":

    st.header("Add Learning App")

    name=st.text_input("App Name")

    category=st.selectbox(
        "Category",
        [
            "Math",
            "Science",
            "Language",
            "Coding",
            "General Learning"
        ]
    )

    age=st.selectbox(
        "Age Group",
        [
            "3-5",
            "6-8",
            "9-12",
            "13+"
        ]
    )

    if st.button("Save"):

        cursor.execute(
        "INSERT INTO apps(name,category,age) VALUES(?,?,?)",
        (name,category,age)
        )

        conn.commit()

        st.success("App Added Successfully")

# -------------------------
# EVALUATION
# -------------------------

elif menu=="Evaluate App":

    st.header("Evaluate Learning App")

    apps=pd.read_sql("SELECT * FROM apps",conn)

    if len(apps)==0:

        st.warning("Please Add Apps First")

    else:

        selected=st.selectbox(
            "Select App",
            apps["name"]
        )

        education=st.slider(
            "Educational Value",
            1,
            10,
            8
        )

        accessibility=st.slider(
            "Accessibility",
            1,
            10,
            8
        )

        usability=st.slider(
            "Ease of Use",
            1,
            10,
            8
        )

        offline=st.slider(
            "Offline Support",
            1,
            10,
            5
        )

        language=st.slider(
            "Language Support",
            1,
            10,
            8
        )

        community=st.slider(
            "Community Relevance",
            1,
            10,
            8
        )

        total=education+accessibility+usability+offline+language+community

        st.metric(
            "Total Score",
            total
        )

        if st.button("Submit Evaluation"):

            cursor.execute("""
            INSERT INTO evaluation
            (
            app,
            education,
            accessibility,
            usability,
            offline,
            language,
            community,
            total
            )

            VALUES(?,?,?,?,?,?,?,?)

            """,

            (
            selected,
            education,
            accessibility,
            usability,
            offline,
            language,
            community,
            total
            )

            )

            conn.commit()

            st.success("Evaluation Saved")

# -------------------------
# RESULTS
# -------------------------

elif menu=="View Results":

    st.header("Evaluation Results")

    df=pd.read_sql("""
    SELECT *
    FROM evaluation
    ORDER BY total DESC
    """,conn)

    if len(df)==0:

        st.warning("No Data")

    else:

        st.dataframe(df,use_container_width=True)

        fig=px.bar(
        df,
        x="app",
        y="total",
        color="total",
        title="App Ranking"
        )

        st.plotly_chart(fig,use_container_width=True)

        pie=px.pie(
        df,
        names="app",
        values="total",
        title="Score Distribution"
        )

        st.plotly_chart(pie,use_container_width=True)

# -------------------------
# DASHBOARD
# -------------------------

else:

    st.header("Dashboard")

    apps=pd.read_sql("SELECT * FROM apps",conn)

    evaluation=pd.read_sql("SELECT * FROM evaluation",conn)

    c1,c2,c3=st.columns(3)

    c1.metric(
    "Total Apps",
    len(apps)
    )

    c2.metric(
    "Total Evaluations",
    len(evaluation)
    )

    if len(evaluation)>0:

        best=evaluation.sort_values(
        "total",
        ascending=False
        ).iloc[0]

        c3.metric(
        "Best App",
        best["app"]
        )

        st.subheader("Top Ranked Apps")

        rank=evaluation.sort_values(
        "total",
        ascending=False
        )

        st.table(rank[["app","total"]])

2026-07-28 18:58:08.378 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-28 18:58:08.380 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-28 18:58:08.529 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-28 18:58:08.530 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-28 18:58:08.531 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-28 18:58:08.533 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-28 18:58:08.536 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [3]:
%%writefile app.py
import streamlit as st
import sqlite3
import pandas as pd
import plotly.express as px

# -------------------------
# DATABASE
# -------------------------

conn = sqlite3.connect("evaluation.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS apps(
id INTEGER PRIMARY KEY AUTOINCREMENT,
name TEXT,
category TEXT,
age TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS evaluation(
id INTEGER PRIMARY KEY AUTOINCREMENT,
app TEXT,
education INTEGER,
accessibility INTEGER,
usability INTEGER,
offline INTEGER,
language INTEGER,
community INTEGER,
total INTEGER
)
""")

conn.commit()

# -------------------------
# PAGE
# -------------------------

st.set_page_config(
page_title="Mobile Learning Evaluation",
layout="wide"
)

st.title("📚 Mobile Learning App Evaluation System")

menu = st.sidebar.selectbox(
"Menu",
[
"Dashboard",
"Add App",
"Evaluate App",
"View Results"
]
)

# -------------------------
# ADD APP
# -------------------------

if menu=="Add App":

    st.header("Add Learning App")

    name=st.text_input("App Name")

    category=st.selectbox(
        "Category",
        [
            "Math",
            "Science",
            "Language",
            "Coding",
            "General Learning"
        ]
    )

    age=st.selectbox(
        "Age Group",
        [
            "3-5",
            "6-8",
            "9-12",
            "13+"
        ]
    )

    if st.button("Save"):

        cursor.execute(
        "INSERT INTO apps(name,category,age) VALUES(?,?,?)",
        (name,category,age)
        )

        conn.commit()

        st.success("App Added Successfully")

# -------------------------
# EVALUATION
# -------------------------

elif menu=="Evaluate App":

    st.header("Evaluate Learning App")

    apps=pd.read_sql("SELECT * FROM apps",conn)

    if len(apps)==0:

        st.warning("Please Add Apps First")

    else:

        selected=st.selectbox(
            "Select App",
            apps["name"]
        )

        education=st.slider(
            "Educational Value",
            1,
            10,
            8
        )

        accessibility=st.slider(
            "Accessibility",
            1,
            10,
            8
        )

        usability=st.slider(
            "Ease of Use",
            1,
            10,
            8
        )

        offline=st.slider(
            "Offline Support",
            1,
            10,
            5
        )

        language=st.slider(
            "Language Support",
            1,
            10,
            8
        )

        community=st.slider(
            "Community Relevance",
            1,
            10,
            8
        )

        total=education+accessibility+usability+offline+language+community

        st.metric(
            "Total Score",
            total
        )

        if st.button("Submit Evaluation"):

            cursor.execute("""
            INSERT INTO evaluation
            (
            app,
            education,
            accessibility,
            usability,
            offline,
            language,
            community,
            total
            )

            VALUES(?,?,?,?,?,?,?,?)

            """,

            (
            selected,
            education,
            accessibility,
            usability,
            offline,
            language,
            community,
            total
            )

            )

            conn.commit()

            st.success("Evaluation Saved")

# -------------------------
# RESULTS
# -------------------------

elif menu=="View Results":

    st.header("Evaluation Results")

    df=pd.read_sql("""
    SELECT *
    FROM evaluation
    ORDER BY total DESC
    """,conn)

    if len(df)==0:

        st.warning("No Data")

    else:

        st.dataframe(df,use_container_width=True)

        fig=px.bar(
        df,
        x="app",
        y="total",
        color="total",
        title="App Ranking"
        )

        st.plotly_chart(fig,use_container_width=True)

        pie=px.pie(
        df,
        names="app",
        values="total",
        title="Score Distribution"
        )

        st.plotly_chart(pie,use_container_width=True)

# -------------------------
# DASHBOARD
# -------------------------

else:

    st.header("Dashboard")

    apps=pd.read_sql("SELECT * FROM apps",conn)

    evaluation=pd.read_sql("SELECT * FROM evaluation",conn)

    c1,c2,c3=st.columns(3)

    c1.metric(
    "Total Apps",
    len(apps)
    )

    c2.metric(
    "Total Evaluations",
    len(evaluation)
    )

    if len(evaluation)>0:

        best=evaluation.sort_values(
        "total",
        ascending=False
        ).iloc[0]

        c3.metric(
        "Best App",
        best["app"]
        )

        st.subheader("Top Ranked Apps")

        rank=evaluation.sort_values(
        "total",
        ascending=False
        )

        st.table(rank[["app","total"]])

Writing app.py


In [4]:
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-07-28 19:03:31.587 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.204.152.59:8501

  Stopping...
^C
